# 🏀 Fantasy NBA Draft Engine — 2026-27 Season
### Full Auction Draft: $200 Budget | 10 Teams | 13 Players Each
**Stats Scored:** PTS_PG, AST_PG, FG3M_PG, REB_PG, STL_PG, BLK_PG, FG_PCT, FT_PCT, DD_PG, PTS_MIN

**Positions Required:** F, G, C, PF, SF, SG, PG (10 starters + 3 bench)

In [15]:
# ─── CELL 1: IMPORTS ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from nba_api.stats.endpoints import leaguegamelog, commonallplayers, commonplayerinfo
from nba_api.stats.static import players as static_players

print("✅ Imports OK")

✅ Imports OK


In [16]:
# ─── CELL 2: FETCH ALL GAME LOGS (2015-16 → 2025-26) ────────────────────────
SEASONS = [
    '2015-16', '2016-17', '2017-18', '2018-19', '2019-20',
    '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26'
]

all_seasons_data = []
print("Fetching game logs — ~30s due to rate limits...")

for season in SEASONS:
    print(f"  → {season}", end="", flush=True)
    try:
        gl = leaguegamelog.LeagueGameLog(
            season=season,
            player_or_team_abbreviation='P',
            season_type_all_star='Regular Season'
        )
        df_s = gl.get_data_frames()[0]
        df_s['SEASON'] = season
        all_seasons_data.append(df_s)
        print(f" [{len(df_s):,} rows]")
        time.sleep(2.5)
    except Exception as e:
        print(f" ERROR: {e}")

df_all_stats = pd.concat(all_seasons_data, ignore_index=True)
print(f"\n✅ Total rows: {len(df_all_stats):,}")
print("Columns:", list(df_all_stats.columns))

Fetching game logs — ~30s due to rate limits...
  → 2015-16 [26,078 rows]
  → 2016-17 [26,139 rows]
  → 2017-18 [26,107 rows]
  → 2018-19 [26,101 rows]
  → 2019-20 [22,393 rows]
  → 2020-21 [23,054 rows]
  → 2021-22 [26,039 rows]
  → 2022-23 [25,895 rows]
  → 2023-24 [26,401 rows]
  → 2024-25 [26,306 rows]
  → 2025-26 [23,645 rows]

✅ Total rows: 278,158
Columns: ['SEASON_ID', 'PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'PLUS_MINUS', 'FANTASY_PTS', 'VIDEO_AVAILABLE', 'SEASON']


In [20]:
# ─── CELL 3: SCRAPE AGE + POSITION (per player) ─────────────────────────────
# Gets BIRTH_DATE and PRIMARY_POSITION for every unique player in dataset.
# ⚠ This takes ~5-10 min depending on roster size. Run once and cache.

unique_ids = df_all_stats['PLAYER_ID'].unique()
print(f"Fetching metadata for {len(unique_ids)} players...")

player_meta = []
errors = []

for i, pid in enumerate(unique_ids):
    print(i)
    try:
        info = commonplayerinfo.CommonPlayerInfo(player_id=pid)
        row = info.get_data_frames()[0].iloc[0]
        player_meta.append({
            'PLAYER_ID': pid,
            'BIRTH_DATE': row.get('BIRTHDATE', None),
            'POSITION': row.get('POSITION', 'G'),   # e.g. "Forward-Guard" or "Center"
            'DRAFT_YEAR': row.get('DRAFT_YEAR', None)
        })
        if i % 50 == 0:
            print(f"  {i}/{len(unique_ids)} done...")
        time.sleep(1.2)
    except Exception as e:
        errors.append(pid)
        time.sleep(1.2)

df_meta = pd.DataFrame(player_meta)
df_meta['BIRTH_DATE'] = pd.to_datetime(df_meta['BIRTH_DATE'], errors='coerce')

# ── Normalize POSITION to fantasy slots ─────────────────────────────────────
# NBA API returns values like: "Guard", "Forward", "Center", "Forward-Guard", etc.
def normalize_position(pos_str):
    """Map NBA API position strings to primary fantasy slot."""
    if not isinstance(pos_str, str):
        return 'G'
    p = pos_str.strip().lower()
    if p in ['center', 'c']:               return 'C'
    if 'center' in p and 'forward' in p:  return 'PF'   # stretch big
    if 'forward-guard' in p:              return 'SF'
    if 'guard-forward' in p:              return 'SG'
    if 'forward' in p:                    return 'PF'
    if 'guard' in p:                      return 'SG'
    return 'G'  # fallback

df_meta['POS_PRIMARY'] = df_meta['POSITION'].apply(normalize_position)

# ── PG detection: use assists-heavy players among guards ────────────────────
# We'll refine PG assignment after stats are computed (high AST_PG guards → PG)

print(f"\n✅ Metadata fetched. Errors: {len(errors)} players skipped.")
df_meta.head()

0


KeyboardInterrupt: 

In [22]:
# ─── CELL 4: CLEAN + FEATURE ENGINEERING ────────────────────────────────────

CURRENT_SEASON = '2025-26'
PREV_SEASON    = '2024-25'
MAX_GAMES      = 82
DRAFT_YEAR_CURRENT = 2025   # Players drafted in 2025 = 2025-26 rookies

def parse_minutes(val):
    """Handle '35:12' and float/int formats."""
    if isinstance(val, str) and ':' in val:
        parts = val.split(':')
        return int(parts[0]) + int(parts[1]) / 60
    try:
        return float(val) if pd.notna(val) else 0.0
    except:
        return 0.0

def clean_and_prep(df):
    df = df.copy()
    df['MIN'] = df['MIN'].apply(parse_minutes)
    dd_cats = ['PTS', 'REB', 'AST', 'STL', 'BLK']
    df['DD'] = ((df[dd_cats] >= 10).sum(axis=1) >= 2).astype(int)
    df['STOCKS'] = df['STL'] + df['BLK']
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    return df

df_clean = clean_and_prep(df_all_stats)
print("✅ Clean done.")

✅ Clean done.


In [23]:
# ─── CELL 5: AGGREGATE STATS (with ROOKIE exception) ────────────────────────
# Standard path: GP >= 15 AND MPG >= 12
# Rookie path:   GP >= 5  AND MPG >= 10  (first season — small sample but valid)

def aggregate_stats(df, label='', is_rookie_pass=False):
    agg_map = {
        'GAME_ID': 'count', 'MIN': 'sum', 'FGM': 'sum', 'FGA': 'sum',
        'FTM': 'sum', 'FTA': 'sum', 'FG3M': 'sum', 'REB': 'sum',
        'AST': 'sum', 'STL': 'sum', 'BLK': 'sum', 'PTS': 'sum', 'DD': 'sum'
    }
    grouped = df.groupby(['PLAYER_ID', 'PLAYER_NAME']).agg(agg_map).reset_index()
    grouped.rename(columns={'GAME_ID': 'GP'}, inplace=True)
    grouped['MPG'] = grouped['MIN'] / grouped['GP']

    if is_rookie_pass:
        # Relaxed filter for first-year players
        grouped = grouped[(grouped['GP'] >= 5) & (grouped['MPG'] >= 10)]
        grouped['IS_ROOKIE_SAMPLE'] = True
    else:
        grouped = grouped[(grouped['GP'] >= 15) & (grouped['MPG'] >= 12)]
        grouped['IS_ROOKIE_SAMPLE'] = False

    for s in ['FG3M', 'REB', 'AST', 'STL', 'BLK', 'PTS', 'DD']:
        grouped[f'{s}_PG'] = grouped[s] / grouped['GP']

    grouped['FG_PCT']  = np.where(grouped['FGA'] > 0, grouped['FGM'] / grouped['FGA'], np.nan)
    grouped['FT_PCT']  = np.where(grouped['FTA'] > 0, grouped['FTM'] / grouped['FTA'], np.nan)
    grouped['PTS_MIN'] = np.where(grouped['MIN'] > 0, grouped['PTS'] / grouped['MIN'], np.nan)

    if label:
        grouped = grouped.add_suffix(f'_{label}')
        grouped.rename(columns={f'PLAYER_ID_{label}': 'PLAYER_ID',
                                  f'PLAYER_NAME_{label}': 'PLAYER_NAME'}, inplace=True)
    return grouped


# ── Identify rookies (players whose draft year matches current season start) ─
rookie_ids = set()
if df_meta is not None:
    draft_year_col = pd.to_numeric(df_meta['DRAFT_YEAR'], errors='coerce')
    rookie_ids = set(df_meta[draft_year_col == DRAFT_YEAR_CURRENT]['PLAYER_ID'].tolist())

df_current = df_clean[df_clean['SEASON'] == CURRENT_SEASON]

# Standard players (not rookies)
df_26_standard = aggregate_stats(
    df_current[~df_current['PLAYER_ID'].isin(rookie_ids)], label='26'
)
# Rookies with relaxed filter
df_26_rookies  = aggregate_stats(
    df_current[df_current['PLAYER_ID'].isin(rookie_ids)], label='26', is_rookie_pass=True
)

df_26 = pd.concat([df_26_standard, df_26_rookies], ignore_index=True)
df_25 = aggregate_stats(df_clean[df_clean['SEASON'] == PREV_SEASON], label='25')

# Hot-streak detector: last 15 games of current season
last_15 = df_current.sort_values('GAME_DATE').groupby('PLAYER_ID').tail(15)
df_late = aggregate_stats(last_15, label='LATE')

print(f"✅ 25-26 pool: {len(df_26)} players  (rookies included: {len(df_26_rookies)})")
print(f"✅ 24-25 pool: {len(df_25)} players")

✅ 25-26 pool: 385 players  (rookies included: 0)
✅ 24-25 pool: 394 players


In [24]:
# ─── CELL 6: RELIABILITY + AGE CALCULATIONS ──────────────────────────────────

# Historical GP reliability (last 3 seasons)
recent_seasons = ['2022-23', '2023-24', '2024-25']
gp_history = (
    df_clean[df_clean['SEASON'].isin(recent_seasons)]
    .groupby(['PLAYER_ID', 'PLAYER_NAME', 'SEASON'])['GAME_ID']
    .count().reset_index()
)
reliability = (
    gp_history.groupby(['PLAYER_ID', 'PLAYER_NAME'])['GAME_ID']
    .mean().reset_index()
    .rename(columns={'GAME_ID': 'AVG_GP'})
)
reliability['RELIABILITY_SCORE'] = (reliability['AVG_GP'] / MAX_GAMES).clip(0.2, 1.0)

# ── Age as of draft day (Oct 2026) ──────────────────────────────────────────
DRAFT_DATE = pd.Timestamp('2026-10-01')
df_meta['AGE'] = ((DRAFT_DATE - df_meta['BIRTH_DATE']).dt.days / 365.25).round(1)

# ── Age factor for predictions ───────────────────────────────────────────────
# Prime window 22-28: neutral. <22: growth multiplier. >32: decay.
def age_factor(age):
    if pd.isna(age):    return 1.0
    if age < 22:        return 1.12   # Rookie/sophomore breakout potential
    if age < 25:        return 1.06   # Early prime approach
    if age <= 28:       return 1.0    # Full prime — no adjustment
    if age <= 31:       return 0.97   # Slight decline possible
    if age <= 33:       return 0.91   # Real decline risk
    return 0.84                        # Veteran fall-off

df_meta['AGE_FACTOR'] = df_meta['AGE'].apply(age_factor)

print("✅ Reliability & age computed.")
print(df_meta[['PLAYER_ID','POSITION','POS_PRIMARY','AGE','AGE_FACTOR']].head(10))

✅ Reliability & age computed.
   PLAYER_ID        POSITION POS_PRIMARY   AGE  AGE_FACTOR
0     200757   Forward-Guard          SF  42.4        0.84
1     201952           Guard          SG  38.3        0.84
2     203145   Guard-Forward          SG  37.3        0.84
3       2760         Forward          PF  44.0        0.84
4     202389          Center           C  40.2        0.84
5     202684  Center-Forward          PF  35.6        0.84
6     203099           Guard          SG  35.4        0.84
7     201569           Guard          SG  37.8        0.84
8     201582          Center           C  38.4        0.84
9     201583         Forward          PF  38.4        0.84


In [25]:
# ─── CELL 7: MASTER MERGE + PREDICTIONS ──────────────────────────────────────

PREDICT_CATS = [
    'PTS_PG', 'AST_PG', 'FG3M_PG', 'REB_PG', 'STL_PG',
    'BLK_PG', 'FG_PCT', 'FT_PCT', 'DD_PG', 'PTS_MIN'
]

df_master = (
    df_26
    .merge(df_25, on=['PLAYER_ID', 'PLAYER_NAME'], how='left')
    .merge(reliability[['PLAYER_ID', 'RELIABILITY_SCORE']], on='PLAYER_ID', how='left')
    .merge(df_late[['PLAYER_ID', 'PTS_PG_LATE', 'PTS_MIN_LATE', 'AST_PG_LATE']], on='PLAYER_ID', how='left')
    .merge(df_meta[['PLAYER_ID', 'POS_PRIMARY', 'AGE', 'AGE_FACTOR', 'DRAFT_YEAR']], on='PLAYER_ID', how='left')
)

# Default reliability for rookies (no 3-year history)
df_master['RELIABILITY_SCORE'] = df_master['RELIABILITY_SCORE'].fillna(0.60)  # rookies = 0.60 baseline
df_master['AGE_FACTOR']        = df_master['AGE_FACTOR'].fillna(1.0)
df_master['POS_PRIMARY']       = df_master['POS_PRIMARY'].fillna('G')

# Flag rookies
df_master['IS_ROOKIE'] = df_master['PLAYER_ID'].isin(rookie_ids)
# Confirm IS_ROOKIE_SAMPLE too
df_master['IS_ROOKIE_SAMPLE'] = df_master['IS_ROOKIE_SAMPLE_26'].fillna(False)

# ── Weighted Predictions ─────────────────────────────────────────────────────
for cat in PREDICT_CATS:
    c26  = f'{cat}_26'
    c25  = f'{cat}_25'
    pred = f'PRED_{cat}'

    # Rookies: no prior season → use 26 only, apply age growth factor
    has_prior = df_master[c25].notna()
    df_master[pred] = np.where(
        has_prior,
        df_master[c26] * 0.70 + df_master[c25] * 0.30,
        df_master[c26]   # rookie: 100% current season data
    )

    # Apply age factor to all predictions
    df_master[pred] = df_master[pred] * df_master['AGE_FACTOR']

    # Rookie confidence penalty (small sample) — shrink 10% toward league mean
    league_mean = df_master[pred].mean()
    df_master.loc[df_master['IS_ROOKIE_SAMPLE'], pred] = (
        df_master.loc[df_master['IS_ROOKIE_SAMPLE'], pred] * 0.85
        + league_mean * 0.15
    )

# ── Late-Season Breakout Boost ────────────────────────────────────────────────
for surge_cat in ['PTS_PG', 'PTS_MIN', 'AST_PG']:
    late_col = f'{surge_cat}_LATE'
    pred_col = f'PRED_{surge_cat}'
    mask = (
        df_master[late_col].notna() &
        (df_master[late_col] > df_master[f'{surge_cat}_26'] * 1.15)
    )
    df_master.loc[mask, pred_col] *= 1.08   # 8% breakout multiplier

print(f"✅ Master built: {len(df_master)} players")
print(f"   Rookies in pool: {df_master['IS_ROOKIE'].sum()}")

✅ Master built: 385 players
   Rookies in pool: 0


In [26]:
# ─── CELL 8: Z-SCORES + COMPOSITE VALUE ──────────────────────────────────────

for cat in PREDICT_CATS:
    col = f'PRED_{cat}'
    mu  = df_master[col].mean()
    sd  = df_master[col].std()
    df_master[f'Z_{cat}'] = (df_master[col] - mu) / sd

# Expected GP = reliability * 82
df_master['EXP_GP'] = (df_master['RELIABILITY_SCORE'] * MAX_GAMES).round(0)

# Risk flag
df_master['RISK_FLAG'] = np.select(
    [df_master['RELIABILITY_SCORE'] >= 0.85,
     df_master['RELIABILITY_SCORE'] >= 0.70,
     df_master['IS_ROOKIE']],
    ['ELITE DURABILITY', 'STABLE', 'ROOKIE RISK'],
    default='HIGH RISK'
)

# Volume-adjusted Z (per-game Z * GP availability ratio)
for cat in PREDICT_CATS:
    df_master[f'V_{cat}'] = df_master[f'Z_{cat}'] * (df_master['EXP_GP'] / MAX_GAMES)

val_cols = [f'V_{cat}' for cat in PREDICT_CATS]
df_master['DRAFT_VALUE_TOTAL'] = df_master[val_cols].sum(axis=1)

# Position-axis composites for visualization
df_master['OFFENSE_AXIS']  = df_master[['Z_PTS_PG','Z_AST_PG','Z_PTS_MIN','Z_FG3M_PG','Z_FT_PCT','Z_STL_PG']].sum(axis=1)
df_master['DEFENSE_AXIS']  = df_master[['Z_REB_PG','Z_FG_PCT','Z_BLK_PG','Z_DD_PG']].sum(axis=1)

# Refine PG detection: guards ranked top-25 in AST_PG → assign PG
guard_mask = df_master['POS_PRIMARY'].isin(['G','SG'])
top_ast_guards = df_master[guard_mask].nlargest(25, 'PRED_AST_PG').index
df_master.loc[top_ast_guards, 'POS_PRIMARY'] = 'PG'

print("✅ Z-scores + value computed.")
print(df_master[['PLAYER_NAME','POS_PRIMARY','AGE','IS_ROOKIE','DRAFT_VALUE_TOTAL']].sort_values('DRAFT_VALUE_TOTAL', ascending=False).head(15))

✅ Z-scores + value computed.
                 PLAYER_NAME POS_PRIMARY   AGE  IS_ROOKIE  DRAFT_VALUE_TOTAL
47              Nikola Jokić           C  31.6      False          18.786306
286        Victor Wembanyama           G   NaN      False          16.374846
123              Luka Dončić          SF  27.6      False          16.076782
103  Shai Gilgeous-Alexander          PG  28.2      False          13.618737
160          Anthony Edwards           G   NaN      False          12.391837
171             Tyrese Maxey          PG   NaN      False          11.358379
221              Josh Giddey          PG   NaN      False          10.931719
208            Jalen Johnson          PG   NaN      False          10.383027
31     Giannis Antetokounmpo          PF  31.8      False          10.369142
87               Bam Adebayo           G   NaN      False          10.249870
219           Alperen Sengun          PG   NaN      False          10.237891
213           Scottie Barnes          PG   NaN 

In [27]:
# ─── CELL 9: AUCTION VALUES ───────────────────────────────────────────────────

NUM_TEAMS           = 10
BUDGET_PER_TEAM     = 200
TOTAL_BUDGET        = NUM_TEAMS * BUDGET_PER_TEAM   # $2,000
ROSTER_SIZE         = 13   # 10 starters + 3 bench
TOTAL_DRAFTED       = NUM_TEAMS * ROSTER_SIZE        # 130 players
MIN_BID             = 1

df_auction = df_master.sort_values('DRAFT_VALUE_TOTAL', ascending=False).reset_index(drop=True)

# Value Above Replacement: 130th player = floor
replacement_value       = df_auction.iloc[TOTAL_DRAFTED - 1]['DRAFT_VALUE_TOTAL']
df_auction['VAR']       = (df_auction['DRAFT_VALUE_TOTAL'] - replacement_value).clip(lower=0)

remaining_budget        = TOTAL_BUDGET - (TOTAL_DRAFTED * MIN_BID)
total_var               = df_auction['VAR'].head(TOTAL_DRAFTED).sum()
df_auction['EST_$']     = (
    (df_auction['VAR'] / total_var) * remaining_budget + MIN_BID
).round(0).astype(int)
df_auction.loc[TOTAL_DRAFTED:, 'EST_$'] = 0

SHOW_COLS = [
    'PLAYER_NAME', 'POS_PRIMARY', 'AGE', 'IS_ROOKIE',
    'EST_$', 'EXP_GP', 'RISK_FLAG', 'DRAFT_VALUE_TOTAL',
    'PRED_PTS_PG', 'PRED_AST_PG', 'PRED_REB_PG',
    'PRED_STL_PG', 'PRED_BLK_PG', 'PRED_FG3M_PG',
    'PRED_FG_PCT', 'PRED_FT_PCT', 'PRED_DD_PG', 'PRED_PTS_MIN'
]

print(f"\n{'─'*70}")
print(f"  2026-27 AUCTION DRAFT BOARD  |  ${BUDGET_PER_TEAM} per team  |  {ROSTER_SIZE} players")
print(f"{'─'*70}")
print(df_auction[SHOW_COLS].head(30).to_string(index=False))


──────────────────────────────────────────────────────────────────────
  2026-27 AUCTION DRAFT BOARD  |  $200 per team  |  13 players
──────────────────────────────────────────────────────────────────────
            PLAYER_NAME POS_PRIMARY  AGE  IS_ROOKIE  EST_$  EXP_GP        RISK_FLAG  DRAFT_VALUE_TOTAL  PRED_PTS_PG  PRED_AST_PG  PRED_REB_PG  PRED_STL_PG  PRED_BLK_PG  PRED_FG3M_PG  PRED_FG_PCT  PRED_FT_PCT  PRED_DD_PG  PRED_PTS_MIN
           Nikola Jokić           C 31.6      False     68    73.0 ELITE DURABILITY          18.786306    25.803072     9.689572    11.638990     1.406869     0.691690      1.669424     0.520594     0.747880    0.768255      0.728284
      Victor Wembanyama           G  NaN      False     59    58.0           STABLE          16.374846    24.223088     3.183583    11.156897     1.075337     3.320240      2.277811     0.496108     0.823521    0.629160      0.799220
            Luka Dončić          SF 27.6      False     58    62.0           STABLE         

In [32]:
# ─── CELL 10: OPTIMAL TEAM BUILDER ───────────────────────────────────────────
# Builds the BEST possible team for a $200 budget with mandatory position slots.
# Positions required (10 starters): PG, SG, SF, PF, C, G, F, F, G, C_or_PF
# + 3 bench (any position, best available value)

# ── Position eligibility map ──────────────────────────────────────────────────
# Each slot and which POS_PRIMARY values can fill it
STARTER_SLOTS = [
    ('PG', ['PG']),
    ('SG', ['SG', 'G']),
    ('SF', ['SF', 'F']),
    ('PF', ['PF', 'F']),
    ('C',  ['C']),
    ('G',  ['PG', 'SG', 'G']),     # UTIL guard
    ('F',  ['SF', 'PF', 'F']),     # UTIL forward
    ('FLEX', ['C', 'PF', 'SF', 'F', 'PG', 'SG', 'G']),     # 2nd UTIL forward
    ('FLEX', ['C', 'PF', 'SF', 'F', 'PG', 'SG', 'G']),     # 2nd UTIL guard
    ('FLEX', ['C', 'PF', 'SF', 'F', 'PG', 'SG', 'G']),  # flex 10th starter
]
BENCH_COUNT = 3

def build_optimal_team(budget=200, pool=None, exclude_ids=None):
    """
    Greedy positional draft within budget.
    Allocates ~85% budget to starters, 15% to bench.
    """
    if pool is None:
        pool = df_auction[df_auction['EST_$'] > 0].copy()
    if exclude_ids:
        pool = pool[~pool['PLAYER_ID'].isin(exclude_ids)]

    STARTER_BUDGET = int(budget * 0.85)
    BENCH_BUDGET   = budget - STARTER_BUDGET

    used_ids = set()
    roster   = []
    remaining = STARTER_BUDGET

    for slot_name, eligible_pos in STARTER_SLOTS:
        candidates = pool[
            pool['POS_PRIMARY'].isin(eligible_pos) &
            ~pool['PLAYER_ID'].isin(used_ids) &
            (pool['EST_$'] <= remaining - (len(STARTER_SLOTS) - len(roster) - 1))
        ].sort_values('DRAFT_VALUE_TOTAL', ascending=False)

        if candidates.empty:
            # Fallback: any affordable player in eligible position
            candidates = pool[
                pool['POS_PRIMARY'].isin(eligible_pos) &
                ~pool['PLAYER_ID'].isin(used_ids) &
                (pool['EST_$'] <= remaining)
            ].sort_values('DRAFT_VALUE_TOTAL', ascending=False)

        if candidates.empty:
            print(f"  ⚠ Could not fill slot {slot_name}")
            continue

        pick = candidates.iloc[0]
        roster.append({**pick.to_dict(), 'SLOT': slot_name, 'ROLE': 'STARTER'})
        used_ids.add(pick['PLAYER_ID'])
        remaining -= int(pick['EST_$'])

    # ── Bench: best remaining value within leftover budget ────────────────────
    bench_pool = pool[
        ~pool['PLAYER_ID'].isin(used_ids) &
        (pool['EST_$'] <= remaining)
    ].sort_values('DRAFT_VALUE_TOTAL', ascending=False)

    for i in range(BENCH_COUNT):
        bench_available = bench_pool[
            ~bench_pool['PLAYER_ID'].isin(used_ids) &
            (bench_pool['EST_$'] <= remaining)
        ]
        if bench_available.empty:
            break
        pick = bench_available.iloc[0]
        roster.append({**pick.to_dict(), 'SLOT': f'BENCH_{i+1}', 'ROLE': 'BENCH'})
        used_ids.add(pick['PLAYER_ID'])
        remaining -= int(pick['EST_$'])

    return pd.DataFrame(roster), remaining


# ── Build YOUR team first (greedy best-value) ────────────────────────────────
my_team, leftover = build_optimal_team(budget=200)

display_cols = ['SLOT', 'ROLE', 'PLAYER_NAME', 'POS_PRIMARY', 'AGE', 'IS_ROOKIE',
                'EST_$', 'EXP_GP', 'RISK_FLAG',
                'PRED_PTS_PG', 'PRED_AST_PG', 'PRED_REB_PG',
                'PRED_STL_PG', 'PRED_BLK_PG', 'DRAFT_VALUE_TOTAL']

print(f"\n{'═'*75}")
print("  🏆  YOUR OPTIMAL TEAM  —  2026-27 FANTASY SEASON")
print(f"{'═'*75}")
print(my_team[display_cols].to_string(index=False))
print(f"\n  Budget used: ${200 - leftover} / $200  |  Leftover: ${leftover}")
print(f"  Total Draft Value: {my_team['DRAFT_VALUE_TOTAL'].sum():.2f}")

  ⚠ Could not fill slot FLEX

═══════════════════════════════════════════════════════════════════════════
  🏆  YOUR OPTIMAL TEAM  —  2026-27 FANTASY SEASON
═══════════════════════════════════════════════════════════════════════════
SLOT    ROLE             PLAYER_NAME POS_PRIMARY  AGE  IS_ROOKIE  EST_$  EXP_GP        RISK_FLAG  PRED_PTS_PG  PRED_AST_PG  PRED_REB_PG  PRED_STL_PG  PRED_BLK_PG  DRAFT_VALUE_TOTAL
  PG STARTER Shai Gilgeous-Alexander          PG 28.2      False     48    73.0 ELITE DURABILITY    30.905171     6.854459     4.445450     1.481133     0.829124          13.618737
  SG STARTER       Victor Wembanyama           G  NaN      False     59    58.0           STABLE    24.223088     3.183583    11.156897     1.075337     3.320240          16.374846
  SF STARTER         Kelly Oubre Jr.          SF 30.8      False      3    59.0           STABLE    14.363807     1.835600     5.080434     1.340493     0.509841           1.734150
  PF STARTER   Giannis Antetokounmpo        

In [33]:
# ─── CELL 11: SIMULATE ALL 10 TEAMS ──────────────────────────────────────────
# Your team picks first. Each opponent gets the next-best available.

all_teams = {}
globally_used = set()

for team_idx in range(NUM_TEAMS):
    budget = 200
    if team_idx == 0:
        team_df, leftover = build_optimal_team(budget=budget, exclude_ids=globally_used)
        label = '🏆 YOUR TEAM'
    else:
        # Opponents use slightly randomized value weights (simulate different strategies)
        opp_pool = df_auction[df_auction['EST_$'] > 0].copy()
        # Add noise to simulate imperfect opponents
        noise = np.random.uniform(0.85, 1.15, len(opp_pool))
        opp_pool['DRAFT_VALUE_TOTAL'] = opp_pool['DRAFT_VALUE_TOTAL'] * noise
        opp_pool = opp_pool.sort_values('DRAFT_VALUE_TOTAL', ascending=False).reset_index(drop=True)
        # Recompute EST_$ for opponent pool
        team_df, leftover = build_optimal_team(budget=budget, pool=opp_pool, exclude_ids=globally_used)
        label = f'Team {team_idx + 1}'

    all_teams[label] = team_df
    globally_used.update(team_df['PLAYER_ID'].tolist())

# ── League Summary Table ─────────────────────────────────────────────────────
league_summary = []
for team_name, tdf in all_teams.items():
    row = {'TEAM': team_name, 'PLAYERS': len(tdf)}
    row['BUDGET_USED'] = tdf['EST_$'].sum()
    row['TOTAL_VALUE'] = tdf['DRAFT_VALUE_TOTAL'].sum().round(2)
    for cat in ['PRED_PTS_PG', 'PRED_AST_PG', 'PRED_REB_PG',
                'PRED_STL_PG', 'PRED_BLK_PG', 'PRED_FG3M_PG']:
        row[cat] = tdf[cat].mean().round(2)
    league_summary.append(row)

df_league = pd.DataFrame(league_summary).sort_values('TOTAL_VALUE', ascending=False)

print("\n" + "═"*90)
print("  📊  LEAGUE OVERVIEW — 10-TEAM SIMULATED DRAFT")
print("═"*90)
print(df_league.to_string(index=False))

  ⚠ Could not fill slot FLEX
  ⚠ Could not fill slot F
  ⚠ Could not fill slot FLEX
  ⚠ Could not fill slot FLEX
  ⚠ Could not fill slot FLEX
  ⚠ Could not fill slot SF
  ⚠ Could not fill slot FLEX
  ⚠ Could not fill slot SF
  ⚠ Could not fill slot FLEX
  ⚠ Could not fill slot SF
  ⚠ Could not fill slot FLEX
  ⚠ Could not fill slot FLEX
  ⚠ Could not fill slot SF
  ⚠ Could not fill slot SF
  ⚠ Could not fill slot C
  ⚠ Could not fill slot SF
  ⚠ Could not fill slot C
  ⚠ Could not fill slot SF
  ⚠ Could not fill slot C
  ⚠ Could not fill slot SF
  ⚠ Could not fill slot C

══════════════════════════════════════════════════════════════════════════════════════════
  📊  LEAGUE OVERVIEW — 10-TEAM SIMULATED DRAFT
══════════════════════════════════════════════════════════════════════════════════════════
       TEAM  PLAYERS  BUDGET_USED  TOTAL_VALUE  PRED_PTS_PG  PRED_AST_PG  PRED_REB_PG  PRED_STL_PG  PRED_BLK_PG  PRED_FG3M_PG
     Team 8       10          170        58.74        20.30       

In [34]:
# ─── CELL 12: CATEGORY WIN PROJECTION ────────────────────────────────────────
# Estimate which categories your team wins vs each opponent

CAT_LABELS = {
    'PRED_PTS_PG':  'PTS/G',
    'PRED_AST_PG':  'AST/G',
    'PRED_FG3M_PG': '3PM/G',
    'PRED_REB_PG':  'REB/G',
    'PRED_STL_PG':  'STL/G',
    'PRED_BLK_PG':  'BLK/G',
    'PRED_FG_PCT':  'FG%',
    'PRED_FT_PCT':  'FT%',
    'PRED_DD_PG':   'DD/G',
    'PRED_PTS_MIN': 'PTS/MIN',
}

my_avgs = {cat: all_teams['🏆 YOUR TEAM'][cat].mean() for cat in CAT_LABELS}

print("\n" + "═"*70)
print("  📈  YOUR TEAM — PROJECTED CATEGORY STANDINGS VS EACH OPPONENT")
print("═"*70)
print(f"{'OPPONENT':<15}", end="")
for lbl in CAT_LABELS.values():
    print(f"{lbl:>8}", end="")
print(f"{'W-L':>6}")
print("-"*70)

for team_name, tdf in all_teams.items():
    if team_name == '🏆 YOUR TEAM':
        continue
    wins = losses = 0
    print(f"{team_name:<15}", end="")
    for cat, lbl in CAT_LABELS.items():
        opp_avg = tdf[cat].mean()
        diff    = my_avgs[cat] - opp_avg
        symbol  = '✓' if diff > 0 else '✗'
        if diff > 0: wins   += 1
        else:        losses += 1
        print(f"{symbol:>8}", end="")
    print(f"{wins}-{losses:>3}")

print("\n Legend: ✓ = category win for you | ✗ = category loss")


══════════════════════════════════════════════════════════════════════
  📈  YOUR TEAM — PROJECTED CATEGORY STANDINGS VS EACH OPPONENT
══════════════════════════════════════════════════════════════════════
OPPONENT          PTS/G   AST/G   3PM/G   REB/G   STL/G   BLK/G     FG%     FT%    DD/G PTS/MIN   W-L
----------------------------------------------------------------------
Team 2                ✗       ✗       ✗       ✓       ✗       ✓       ✓       ✗       ✓       ✗4-  6
Team 3                ✗       ✗       ✓       ✗       ✓       ✓       ✓       ✗       ✗       ✓5-  5
Team 4                ✗       ✗       ✓       ✗       ✓       ✓       ✗       ✗       ✗       ✓4-  6
Team 5                ✗       ✗       ✗       ✓       ✗       ✓       ✓       ✗       ✓       ✗4-  6
Team 6                ✗       ✗       ✗       ✓       ✓       ✓       ✓       ✗       ✗       ✗4-  6
Team 7                ✗       ✗       ✗       ✓       ✓       ✓       ✓       ✗       ✓       ✓6-  4
Team 8         

In [35]:
# ─── CELL 13: SAVE OUTPUT ─────────────────────────────────────────────────────

with pd.ExcelWriter('fantasy_draft_2026_27.xlsx', engine='openpyxl') as writer:
    df_auction[SHOW_COLS].head(150).to_excel(writer, sheet_name='Big Board', index=False)
    my_team[display_cols].to_excel(writer, sheet_name='Your Optimal Team', index=False)
    df_league.to_excel(writer, sheet_name='League Overview', index=False)

    for team_name, tdf in all_teams.items():
        sheet = team_name.replace('🏆 ', '').replace(' ', '_')[:31]
        tdf[display_cols].to_excel(writer, sheet_name=sheet, index=False)

print("✅ Saved: fantasy_draft_2026_27.xlsx")

ModuleNotFoundError: No module named 'openpyxl'